# Week 43: Calibrated Abstention Extension

In [ ]:
import csv
import json
import os
from pathlib import Path
from time import perf_counter

from ku_nlp_group_25_project.extensions.calibrated_abstention.calibrated_abstention_extension import CalibratedAbstentionExtension
from ku_nlp_group_25_project.extensions.calibrated_abstention.score_builder import ScoreBuilder
from ku_nlp_group_25_project.utilities import load_dataset

MODEL_NAME = "outputs/week41-qa"
SEED = 13

results_dir = Path.cwd().parent / "results" / "week_43" / "calibrated_abstention_extension"
results_dir.mkdir(exist_ok=True, parents=True)

training_data, validation_data = load_dataset()
validation_data = validation_data.loc[
    validation_data["lang"].isin(["ar", "ko", "te"]),
].reset_index(drop=True)

## 0. Compile and Cache Scores

In [ ]:
score_path = results_dir / "validation_scores.csv"

# get the scores if they are cached
if Path(score_path).exists():
    scores = ScoreBuilder.read_scores_from_path(score_path)
else:
    scorer = ScoreBuilder(
        model_name=MODEL_NAME,
        maximum_answer_tokens=30,
        maximum_length=384,
        stride=128,
    )
    start = perf_counter()
    scores = scorer.scores(
        validation_data=validation_data,
    )

    print(f"Scoring time: {perf_counter() - start:.2f} seconds")

    ScoreBuilder.write_scores_to_path(scores, score_path)

## 1. Fit the Decision Rules

In [ ]:
model = CalibratedAbstentionExtension(
    bootstrap_samples=2000,
    folds=5,
    seed=SEED,
    target_recall=0.90,
)
start = perf_counter()
result = model.evaluate(
    scores=scores,
)

print(f"Fitting time: {perf_counter() - start:.6f} seconds")

## 2. Results

In [ ]:
with open(os.path.join(results_dir, "metrics.csv"), "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["rule"] + list(next(iter(result.metrics.values())).to_dict()))
    writer.writeheader()

    for rule, metrics in result.metrics.items():
        writer.writerow({"rule": rule, **metrics.to_dict()})

with open(os.path.join(results_dir, "metrics_by_language.csv"), "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["rule", "language"] + list(next(iter(next(iter(result.metrics_by_language.values())).values())).to_dict())
    )
    writer.writeheader()

    for rule, per_language in result.metrics_by_language.items():
        for language, metrics in per_language.items():
            writer.writerow({"rule": rule, "language": language, **metrics.to_dict()})

with open(os.path.join(results_dir, "thresholds.json"), "w", encoding="utf-8") as handle:
    json.dump(
        {rule: thresholds.to_dict() for rule, thresholds in result.thresholds.items()},
        handle,
        indent=2,
    )

with open(os.path.join(results_dir, "calibration.json"), "w", encoding="utf-8") as handle:
    json.dump(result.calibration.to_dict(), handle, indent=2)

with open(os.path.join(results_dir, "transitions.json"), "w", encoding="utf-8") as handle:
    json.dump(
        {rule: transitions.to_dict() for rule, transitions in result.transitions.items()},
        handle,
        indent=2,
    )

with open(os.path.join(results_dir, "summary.json"), "w", encoding="utf-8") as handle:
    json.dump(result.to_dict(), handle, indent=2)

### 2.1. Overall Results

In [ ]:
for rule, metrics in result.metrics.items():
    print(
        f"{rule:26s} macro-F1 {metrics.macro_f1:.3f} | "
        f"F1 ans {metrics.answerable_f1:.3f} | F1 unans {metrics.unanswerable_f1:.3f} | "
        f"EM {metrics.exact_match:.3f} | F1 {metrics.token_f1:.3f} | "
        f"coverage {metrics.answerable_coverage:.3f} | abstention recall {metrics.abstention_recall:.3f}"
    )

print("\nFitted cutoffs:")

for rule, thresholds in result.thresholds.items():
    print(f"{rule:26s} cutoff {thresholds.cutoff:+.3f} temperature {thresholds.temperature:.3f}")

    for language, cutoff in thresholds.language_cutoffs.items():
        print(f"{language:>28s} cutoff {cutoff:+.3f}")

### 2.2. Per-Language Results

In [ ]:
for rule in ("A_global_threshold", "B_per_language_thresholds"):
    print(f"\n{rule}")
    for language, metrics in result.metrics_by_language[rule].items():
        print(
            f"  {language}: n {metrics.count} | macro-F1 {metrics.macro_f1:.3f} | "
            f"F1 ans {metrics.answerable_f1:.3f} | F1 unans {metrics.unanswerable_f1:.3f} | "
            f"coverage {metrics.answerable_coverage:.3f} | abstention recall {metrics.abstention_recall:.3f}"
        )

### 2.3. Paired Transitions

In [ ]:
for rule, transitions in result.transitions.items():
    print(
        f"{rule}: rescued abstentions {transitions.rescued_abstentions}, "
        f"broken abstentions {transitions.broken_abstentions}, "
        f"rescued answers {transitions.rescued_answers}, "
        f"broken answers {transitions.broken_answers}, unchanged {transitions.unchanged}"
    )

print(
    f"\nPaired macro-F1 difference (B - A): {result.comparison.difference:+.4f} "
    f"[{result.comparison.lower:+.4f}, {result.comparison.upper:+.4f}], "
    f"no improvement in {result.comparison.not_better_fraction:.1%} of replicates"
)

### 2.4. Calibration Diagnostics

In [ ]:
print(f"Temperature: {result.calibration.temperature:.3f}")
print(
    f"Expected calibration error: {result.calibration.expected_calibration_error:.4f} -> "
    f"{result.calibration.calibrated_expected_calibration_error:.4f}"
)
print(f"Average risk (AURC): {result.calibration.average_risk:.4f}")

for language, average_risk in result.calibration.average_risk_by_language.items():
    print(f"  {language}: {average_risk:.4f}")

print(f"Feature combiner AUC: {result.calibration.feature_area_under_roc:.3f}")
print(
    "\nMacro-F1 of the pooled and temperature-scaled rules: "
    f"{result.metrics['A_global_threshold'].macro_f1:.4f} vs "
    f"{result.metrics['C_temperature_scaled'].macro_f1:.4f}"
)

### 2.5. Error Inspection

In [ ]:
baseline = result.decisions["A_global_threshold"]
candidate = result.decisions["B_per_language_thresholds"]

rescued = [
    score for score, was_answered, is_answered in zip(scores, baseline, candidate)
    if was_answered and not is_answered and not score.answerable
]
broken = [
    score for score, was_answered, is_answered in zip(scores, baseline, candidate)
    if not was_answered and is_answered and score.answerable
]

print("Rescued abstentions:")
for score in rescued[:5]:
    print(f"  {score.language} margin {score.margin:+.3f} span {score.predicted_span[:60]!r}")

print("\nBroken abstentions:")
for score in broken[:5]:
    print(f"  {score.language} margin {score.margin:+.3f} gold {score.gold_answer[:60]!r}")